## Imports and Data Prep

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
import matplotlib.pyplot as plt
from letterboxd_utils import get_user_diary, clean_diary_data, remove_duplicate_movies, linear_regression_numpy, predict_linear_regression, accuracy_within_half_star

/var/folders/sh/9y2x7hyx76bghklsygtm6m000000gn/T/ipykernel_72017/3899210698.py:2: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was not found to be installed on your system.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import pandas as pd


In [2]:
username = 'kmgrimes'
df = get_user_diary(username)
df = clean_diary_data(df)
df['Released'] = pd.to_numeric(df['Released'])
df['Day'] = pd.to_numeric(df['Day'])
df = remove_duplicate_movies(df)

wicked
the-hangover
dont-move
piece-by-piece
carmen-jones
les-miserables
office-space
rio
scooby-doo-camp-scare
island-of-lost-souls
the-idea-of-you
sing-sing
rebel-ridge
beetlejuice-beetlejuice
the-holdovers
bottoms
dicks-the-musical
the-accountant
didi
it-ends-with-us
minions-the-rise-of-gru
the-bad-guys
princess-mononoke
porco-rosso
deadpool-wolverine
best-in-show
maxxxine
now-you-see-me
twisters
red-white-royal-blue
kinds-of-kindness
a-quiet-place-day-one
flushed-away
five-nights-at-freddys
jaws
everybody-wants-some
pearl
inside-out-2
holes
am-i-ok
the-kissing-booth-2
anatomy-of-a-fall
the-devil-wears-prada
eighth-grade
kill-bill-vol-1
anyone-but-you
the-greatest-night-in-pop
but-im-a-cheerleader
small-town-gay-bar
dune-part-two
wonka
dune
all-of-us-strangers
the-meg
notting-hill
poor-things
mamma-mia
black-swan
theater-camp
asteroid-city
saltburn
carol
the-hunger-games-the-ballad-of-songbirds-snakes
angel-falls-christmas
renaissance-a-film-by-beyonce
exmas
bottoms
night-at-the-mus

In [3]:
df.head()

,Month,Day,Film,Released,Ratings,Genres,Year
0,11,21,Wicked,2024,4.5,Thriller,2024
1,11,20,The Hangover,2009,4.0,Comedy,2024
2,11,18,Don't Move,2024,2.0,"Drama, Romance",2024
3,10,23,Piece By Piece,2024,3.0,Documentary,2024
4,10,10,Carmen Jones,1954,4.0,"Drama, Romance",2024


## One-Hot Genres Vector With Release Date to Predict Rating

In [4]:
# Handle missing genres and create a list of unique genres
df['Genres'] = df['Genres'].fillna('None')

all_genres = []
for genres in df['Genres']:
    for genre in genres.split(', '):
        if genre not in all_genres:
            all_genres.append(genre)
unique_genres = sorted(list(set(all_genres)))
num_genres = len(unique_genres)
print(all_genres)

['Thriller', 'Comedy', 'Drama', 'Romance', 'Documentary', 'History', 'Crime', 'Mystery', 'Family', 'Animation', 'Horror', 'Science Fiction', 'Action', 'Fantasy', 'None', 'Adventure', 'Music', 'TV Movie', 'Western']


In [5]:
# transferring to numpy arrays instead of pandas dfs
release_year = df['Released'].values.reshape(-1, 1)  # to a column vector
genre_matrix = np.zeros((len(df), num_genres)) # intialize a matrix of zeros

[[2024]
 [2009]
 [2024]
 [2024]
 [1954]
 [2012]
 [1999]
 [2011]
 [2010]
 [1932]
 [2024]
 [2023]
 [2024]
 [2024]
 [2023]
 [2023]
 [2023]
 [2016]
 [2024]
 [2024]
 [2022]
 [2022]
 [1997]
 [1992]
 [2024]
 [2000]
 [2024]
 [2013]
 [2024]
 [2023]
 [2024]
 [2024]
 [2006]
 [2023]
 [1975]
 [2016]
 [2022]
 [2024]
 [2003]
 [2022]
 [2020]
 [2023]
 [2006]
 [2018]
 [2003]
 [2023]
 [2024]
 [1999]
 [2006]
 [2024]
 [2023]
 [2021]
 [2023]
 [2018]
 [1999]
 [2023]
 [2008]
 [2010]
 [2023]
 [2023]
 [2023]
 [2015]
 [2023]
 [2021]
 [2023]
 [2023]
 [2023]
 [2006]
 [2022]
 [2023]
 [2023]
 [2012]
 [2023]
 [2009]
 [1999]
 [2010]
 [2012]
 [2023]
 [2018]
 [2023]
 [2023]
 [1997]
 [2001]
 [2015]
 [1941]
 [2009]
 [2018]
 [2019]
 [2023]
 [2023]
 [1986]
 [2020]
 [2022]
 [1964]
 [2016]
 [2022]
 [1992]
 [1985]
 [2013]
 [2009]
 [2004]
 [2012]
 [1985]
 [2008]
 [2002]
 [2008]
 [1995]
 [2015]
 [2014]
 [2013]
 [2012]
 [2016]
 [1966]
 [2006]
 [1942]
 [1975]
 [2015]
 [2006]
 [2016]
 [2016]
 [2021]
 [2009]
 [2013]
 [2022]
 [2001]


In [6]:
# One-Hot Encoding for Genres (Feature Matrix)
for i, genres in enumerate(df['Genres']):
    for genre in genres.split(', '):
        j = unique_genres.index(genre)
        genre_matrix[i, j] = 1

In [7]:
# Combine Release Year and Genre Matrix
X = np.concatenate((release_year, genre_matrix), axis=1) #concatenate the release year and the genres together
y = df['Ratings'].values

In [8]:
print("Features Matrix (X) shape:", X.shape)
print("Target Variable (y) shape:", y.shape)
print("\nFeatures (X):")
print(X[2:5])
print("\nTarget Variable (y):")
print(y)

Features Matrix (X) shape: (181, 20)
Target Variable (y) shape: (181,)

Features (X):
[[2.024e+03 0.000e+00 0.000e+00 0.000e+00 0.000e+00 0.000e+00 0.000e+00
  1.000e+00 0.000e+00 0.000e+00 0.000e+00 0.000e+00 0.000e+00 0.000e+00
  0.000e+00 1.000e+00 0.000e+00 0.000e+00 0.000e+00 0.000e+00]
 [2.024e+03 0.000e+00 0.000e+00 0.000e+00 0.000e+00 0.000e+00 1.000e+00
  0.000e+00 0.000e+00 0.000e+00 0.000e+00 0.000e+00 0.000e+00 0.000e+00
  0.000e+00 0.000e+00 0.000e+00 0.000e+00 0.000e+00 0.000e+00]
 [1.954e+03 0.000e+00 0.000e+00 0.000e+00 0.000e+00 0.000e+00 0.000e+00
  1.000e+00 0.000e+00 0.000e+00 0.000e+00 0.000e+00 0.000e+00 0.000e+00
  0.000e+00 1.000e+00 0.000e+00 0.000e+00 0.000e+00 0.000e+00]]

Target Variable (y):
[4.5 4.  2.  3.  4.  4.5 3.  3.  4.  0.  3.  5.  1.5 3.5 4.  5.  2.5 3.
 4.  2.  4.  1.5 4.  5.  3.  3.5 3.  3.5 4.  3.  2.5 4.  2.5 4.5 3.  2.5
 4.  3.5 4.5 2.5 5.  4.  3.5 3.5 3.  2.  3.5 4.  0.  4.5 4.5 4.  4.  3.
 3.5 4.  4.5 4.  4.5 4.  4.  4.5 3.5 1.5 5.  2.5 5.  

#### Splitting the data 60-20-20, train-validation-test

In [9]:
indices = np.arange(X.shape[0]) #create an array of numbers representing the index of the row
np.random.seed(42) #setting a random seed for reproducibility
np.random.shuffle(indices) #shuffles the indices in place

In [10]:
train_cutoff = int(0.6 * len(indices)) #gets the first 60% of the indices
val_cutoff = int(0.8 * len(indices)) #gets the next 20% of the indices after the training cutoff

train_indices = indices[:train_cutoff] #indices for the training set
val_indices = indices[train_cutoff:val_cutoff] #indices for the validation set
test_indices = indices[val_cutoff:] #indices for the test set

In [11]:
X_train, y_train = X[train_indices], y[train_indices] #training set
X_val, y_val = X[val_indices], y[val_indices] #validation set
X_test, y_test = X[test_indices], y[test_indices] #test set

### linear regression defined in the utils file, training portion

In [12]:
# training
weights, intercept = linear_regression_numpy(X_train, y_train)

In [13]:
# predictions on validation and test sets
y_val_pred_np = predict_linear_regression(X_val, weights, intercept)
y_test_pred_np = predict_linear_regression(X_test, weights, intercept)

In [14]:
mse_val_np = mean_squared_error(y_val, y_val_pred_np)
r2_val_np = r2_score(y_val, y_val_pred_np)
mse_test_np = mean_squared_error(y_test, y_test_pred_np)
r2_test_np = r2_score(y_test, y_test_pred_np)

print(f"NumPy - Validation MSE: {mse_val_np:.2f}, R-squared: {r2_val_np:.2f}")
print(f"NumPy - Test MSE: {mse_test_np:.2f}, R-squared: {r2_test_np:.2f}")

NumPy - Validation MSE: 1.10, R-squared: -0.77
NumPy - Test MSE: 2.02, R-squared: -0.44


In [15]:
accuracy_val = accuracy_within_half_star(y_val, y_val_pred_np)
accuracy_test = accuracy_within_half_star(y_test, y_test_pred_np)
print(f"NumPy - Validation Accuracy (within 0.5 stars): {accuracy_val:.2f}")
print(f"NumPy - Test Accuracy (within 0.5 stars): {accuracy_test:.2f}")

NumPy - Validation Accuracy (within 0.5 stars): 0.19
NumPy - Test Accuracy (within 0.5 stars): 0.38


Results are pretty bad as the baseline of random guessing with + or - 0.5 stars is 30% accuracy, probably should try a more complex model with the limited data we have as basic linear regression isn't producing a great result here.

maybe incorporating imdb ratings and budget could help and a more complex model, here is what an AI has to say about the results:

Yes, for a proof of concept, even a "terrible" result like this is perfectly fine, especially given the limitations you've identified. The purpose of a proof of concept is not necessarily to achieve stellar performance but to demonstrate that:

Your pipeline works: You've shown you can collect, clean, process, and feed data into a model.
You can implement and evaluate a model: You've demonstrated your understanding of linear regression (and in a way that fulfills the NumPy requirement, which is great).
You can analyze results and identify limitations/next steps: This is where your reflections on the poor performance and ideas for improvement are crucial.

Focus your discussion on why the results are bad and what you can do to improve them. Here are some key points to emphasize:

Limited Features: As you mentioned, genre and release date are likely insufficient predictors of movie ratings. User preferences are complex and depend on many factors (plot, actors, director, mood, etc.). Explicitly stating this limitation demonstrates your understanding.

Small Dataset: 170 movies, while a good start, are a relatively small dataset for machine learning, especially with many genres. Models need sufficient data to learn complex patterns. Highlight this as a key constraint.

Multicollinearity: Discuss how the one-hot encoded genre features likely introduced multicollinearity, which you addressed with regularization. This shows you understand potential issues with this type of encoding.

Baseline Comparison: Calculate the baseline accuracy (within ±0.5 stars) for predicting the mean rating. If your model's accuracy is even slightly better, you can highlight that as a small win. If not, acknowledge that the model doesn't outperform the simplest baseline and explain why.

Future Improvements: This is the most important part. Clearly outline how you would improve the model:

More Data: Emphasize the need for a larger, more diverse dataset.

Rich Features: Discuss incorporating IMDb ratings, budget, cast/crew information, user reviews, etc. Explain how these could be obtained and preprocessed.

Feature Engineering: Consider interaction terms (e.g., genre x release year), or creating derived features (e.g., a "popularity" score based on various factors).

More Advanced Models: Mention K-Nearest Neighbors, decision trees, or other suitable algorithms as potential next steps. Explain why these might be better suited to this task than linear regression.